In [38]:
print("ram ram")

ram ram


In [39]:
import json
import requests
import pandas as pd
import urllib3
from requests.exceptions import ConnectionError, Timeout, RequestException

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://localhost:7204"
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"
TOKEN = None  # set this if your backend requires auth

headers = {"Accept": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"


def fetch_swagger(url, headers=None, verify=False):
    response = requests.get(url, headers=headers, verify=verify, timeout=15)
    response.raise_for_status()
    return response.json()

try:
    swagger = fetch_swagger(SWAGGER_URL, headers=headers)
    print("Swagger loaded successfully")
except Exception as e:
    print(f"Could not load swagger: {e}")
    swagger = {"paths": {}}

Swagger loaded successfully


In [40]:
def parse_json_if_possible(value):
    if not value:
        return None
    if isinstance(value, dict) or isinstance(value, list):
        return value
    try:
        return json.loads(value)
    except Exception:
        return value

rows = []
for path, methods in swagger.get("paths", {}).items():
    for method, details in methods.items():
        if method.lower() not in {"get", "post", "put", "patch", "delete"}:
            continue

        metadata = {}
        description = parse_json_if_possible(details.get("description"))
        if isinstance(description, dict):
            metadata.update(description)
        elif isinstance(description, str):
            metadata["description_text"] = description

        ext = details.get("x-metadata", {})
        if isinstance(ext, dict):
            metadata.update(ext)

        tool_name = (
            metadata.get("tool_name")
            or details.get("operationId")
            or f"{method.upper()}_{path.replace('/', '_')}"
        )

        rows.append({
            "tool_name": tool_name,
            "path": path,
            "method": method.upper(),
            "description": metadata.get("description") or details.get("summary") or "",
            "business_domain": metadata.get("business_domain") or "Unassigned",
            "recommended_agents": metadata.get("recommended_agents", []),
            "restricted_agents": metadata.get("restricted_agents", []),
            "agent_accessible": bool(metadata.get("agent_accessible", True)),
            "auto_register": bool(metadata.get("auto_register", True)),
            "operation_type": metadata.get("operation_type") or method.upper(),
            "risk_level": metadata.get("risk_level") or "MEDIUM",
            "priority": int(metadata.get("priority", 100)),
        })

backend_api_df = pd.DataFrame(rows)
backend_api_df.head()

,tool_name,path,method,description,business_domain,recommended_agents,restricted_agents,agent_accessible,auto_register,operation_type,risk_level,priority
0,get_status,/api/v1/account,GET,Executes GetStatus. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
1,create_user_profile,/api/v1/account/user-creation,POST,Executes CreateUserProfile. Usable by AccountA...,Unassigned,[],[],False,True,POST,MEDIUM,100
2,get_roles,/api/v1/account/roles,GET,Executes GetRoles. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
3,get_menus,/api/v1/account/menu,GET,Executes GetMenus. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100
4,user_by_id,/api/v1/account/user-by-id/{id},GET,Executes UserById. Usable by AccountApi agents.,Unassigned,[],[],True,True,GET,MEDIUM,100


In [41]:
usable_api_df = backend_api_df[
    (backend_api_df["agent_accessible"] == True) &
    (backend_api_df["auto_register"] == True)
].copy()

usable_api_df = usable_api_df.sort_values(["business_domain", "priority", "tool_name"]).reset_index(drop=True)
print(f"Total discovered APIs: {len(backend_api_df)}")
print(f"Usable APIs for agentic pipelines: {len(usable_api_df)}")

usable_api_df[["tool_name", "business_domain", "operation_type", "priority", "agent_accessible", "auto_register"]].head(20)

Total discovered APIs: 670
Usable APIs for agentic pipelines: 362


,tool_name,business_domain,operation_type,priority,agent_accessible,auto_register
0,GET__api_v1_wiptraceability_getIncompleteJumbo...,Unassigned,GET,100,True,True
1,bom,Unassigned,GET,100,True,True
2,check_bag_status,Unassigned,GET,100,True,True
3,check_r_m_bag_staus,Unassigned,GET,100,True,True
4,download,Unassigned,GET,100,True,True
5,exceptions,Unassigned,GET,100,True,True
6,fore_cast_date_validation,Unassigned,GET,100,True,True
7,gate_by_locatio,Unassigned,GET,100,True,True
8,generate_q_r_code,Unassigned,GET,100,True,True
9,get,Unassigned,GET,100,True,True


In [42]:
def build_agent_definitions(api_df):
    agents = []
    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue
        agents.append({
            "agent_name": f"{domain} Agent",
            "business_domain": domain,
            "tool_count": len(group),
            "tools": group["tool_name"].tolist(),
            "registration_mode": "rule_based",
        })
    return pd.DataFrame(agents)

agent_registry_df = build_agent_definitions(usable_api_df)
agent_registry_df

,agent_name,business_domain,tool_count,tools,registration_mode
0,Unassigned Agent,Unassigned,362,[GET__api_v1_wiptraceability_getIncompleteJumb...,rule_based


In [43]:
def build_rule_based_pipeline_registry(api_df):
    pipeline_rows = []
    pipeline_graph = {}

    for domain, group in api_df.groupby("business_domain", sort=True):
        if group.empty:
            continue

        ordered_group = group.sort_values(["priority", "tool_name"], ascending=[True, True]).reset_index(drop=True)
        selected_tools = ordered_group.head(5).copy()

        agent_name = f"{domain} Agent"
        pipeline_name = f"{domain.lower().replace(' ', '_')}_pipeline"

        graph_steps = []
        for idx, row in selected_tools.iterrows():
            step_name = f"step_{idx + 1}"
            step_id = f"{pipeline_name}_{step_name}"
            depends_on = [] if idx == 0 else [f"{pipeline_name}_step_{idx}"]
            graph_steps.append({
                "step_id": step_id,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "depends_on": depends_on,
            })

            pipeline_rows.append({
                "pipeline_name": pipeline_name,
                "agent_name": agent_name,
                "business_domain": domain,
                "step_order": idx + 1,
                "step_name": step_name,
                "tool_name": row["tool_name"],
                "path": row["path"],
                "operation_type": row["operation_type"],
                "priority": int(row["priority"]),
                "uses_llm": False,
                "reason": "metadata_ordered_rule_based",
            })

        pipeline_graph[agent_name] = {
            "business_domain": domain,
            "pipeline_name": pipeline_name,
            "steps": graph_steps,
        }

    return pd.DataFrame(pipeline_rows), pipeline_graph


pipeline_registry_df, pipeline_graph = build_rule_based_pipeline_registry(usable_api_df)
pipeline_registry_df.head(15)

,pipeline_name,agent_name,business_domain,step_order,step_name,tool_name,path,operation_type,priority,uses_llm,reason
0,unassigned_pipeline,Unassigned Agent,Unassigned,1,step_1,GET__api_v1_wiptraceability_getIncompleteJumbo...,/api/v1/wiptraceability/getIncompleteJumboBag/...,GET,100,False,metadata_ordered_rule_based
1,unassigned_pipeline,Unassigned Agent,Unassigned,2,step_2,bom,/api/v1/planning/bom/{productCode},GET,100,False,metadata_ordered_rule_based
2,unassigned_pipeline,Unassigned Agent,Unassigned,3,step_3,check_bag_status,/api/v1/wiptraceability/check-bag-status/{bagN...,GET,100,False,metadata_ordered_rule_based
3,unassigned_pipeline,Unassigned Agent,Unassigned,4,step_4,check_r_m_bag_staus,/api/v1/wiptraceability/rm-bag/{bagName},GET,100,False,metadata_ordered_rule_based
4,unassigned_pipeline,Unassigned Agent,Unassigned,5,step_5,download,/api/v1/attachment/downaload-by-id/{id},GET,100,False,metadata_ordered_rule_based


In [44]:
import pandas as pd
from sqlalchemy import create_engine, text

In [45]:
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321"
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

In [46]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT version();"))
        print(result.fetchone()[0])
        print("\n✅ PostgreSQL Connected Successfully")
except Exception as e:
    print(e)

PostgreSQL 16.14, compiled by Visual C++ build 1944, 64-bit

✅ PostgreSQL Connected Successfully


In [47]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
"""

tables = pd.read_sql(query, engine)
tables

,table_name
0,agent_execution_logs
1,agent_tool_mapping
2,agentic_pipline
3,alert_master
4,api_logs
5,app_authentication
6,app_connection_table
7,app_table
8,app_version_history
9,approval_history


In [48]:
query = """
SELECT
table_name,
column_name,
data_type
FROM information_schema.columns
WHERE table_schema='public'
ORDER BY table_name, ordinal_position;
"""

columns = pd.read_sql(query, engine)
columns

,table_name,column_name,data_type
0,agent_execution_logs,execution_id,uuid
1,agent_execution_logs,agentic_id,character varying
2,agent_execution_logs,started_at,timestamp with time zone
3,agent_execution_logs,completed_at,timestamp with time zone
4,agent_execution_logs,status,character varying
...,...,...,...
177,workflow_history,input_state,jsonb
178,workflow_history,output_state,jsonb
179,workflow_history,approval_status,character varying
180,workflow_history,customer_id,integer


In [49]:
query = """
SELECT COUNT(*)
FROM information_schema.tables
WHERE table_schema='public';
"""

pd.read_sql(query, engine)

,count
0,26


In [50]:
import requests
import pandas as pd
import json
from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import JSONB

# -----------------------------
# Database Configuration
# -----------------------------
DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321"
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# Swagger URL
# -----------------------------
swagger_url = "https://localhost:7204/swagger/v1/swagger.json"

swagger = requests.get(
    swagger_url,
    verify=False
).json()

rows = []

for endpoint, methods in swagger["paths"].items():

    for method, details in methods.items():

        description = details.get("description")

        # Parse JSON description if possible
        try:
            description_json = json.loads(description) if description else {}
        except:
            description_json = {}

        rows.append({

            "tool_name": description_json.get("tool_name"),

            "method": method.upper(),

            "end_point": endpoint,

            "summary": details.get("summary"),

            "description": description_json.get("description"),

            "agent_accessible": description_json.get(
                "agent_accessible", False
            ),

            "auto_register": description_json.get(
                "auto_register", False
            ),

            "authentication": bool(details.get("security")),

            "request_body": "requestBody" in details,

            "parameter_count": len(
                details.get("parameters", [])
            ),

            "raw_json": description_json
        })

df = pd.DataFrame(rows)

print(df.head())


             tool_name method                        end_point summary  \
0           get_status    GET                  /api/v1/account    None   
1  create_user_profile   POST    /api/v1/account/user-creation    None   
2            get_roles    GET            /api/v1/account/roles    None   
3            get_menus    GET             /api/v1/account/menu    None   
4           user_by_id    GET  /api/v1/account/user-by-id/{id}    None   

                                         description  agent_accessible  \
0   Executes GetStatus. Usable by AccountApi agents.              True   
1  Executes CreateUserProfile. Usable by AccountA...             False   
2    Executes GetRoles. Usable by AccountApi agents.              True   
3    Executes GetMenus. Usable by AccountApi agents.              True   
4    Executes UserById. Usable by AccountApi agents.              True   

   auto_register  authentication  request_body  parameter_count  \
0           True           False         Fa

In [51]:
import requests

BASE_URL = "https://localhost:7204"
TOKEN = "eyJhbGciOiJodHRwOi8vd3d3LnczLm9yZy8yMDAxLzA0L3htbGRzaWctbW9yZSNobWFjLXNoYTI1NiIsInR5cCI6IkpXVCJ9.eyJodHRwOi8vc2NoZW1hcy54bWxzb2FwLm9yZy93cy8yMDA1LzA1L2lkZW50aXR5L2NsYWltcy9uYW1lIjoiNTEiLCJodHRwOi8vc2NoZW1hcy54bWxzb2FwLm9yZy93cy8yMDA1LzA1L2lkZW50aXR5L2NsYWltcy9uYW1laWRlbnRpZmllciI6IklJSU9UIFRlYW0iLCJlbWFpbCI6IklJSU9UIFRlYW0iLCJFbXBsb3llZU5hbWUiOiIiLCJjb21wYW55SWQiOiIxMyIsImRlcGFydG1lbnQiOiIiLCJmdW5jdGlvbklkIjoiMCIsImRlcGFydG1lbnRJZCI6IjAiLCJlbXBsb3llZUlkIjoiIiwiVXNlcklkIjoiNTEiLCJodHRwOi8vc2NoZW1hcy5taWNyb3NvZnQuY29tL3dzLzIwMDgvMDYvaWRlbnRpdHkvY2xhaW1zL3JvbGUiOlsiU0FMRVNfVEVBTSIsIlNGR01hbmFnZXIiLCJEaXNwYXRjaFRlYW0iLCJDb21wYW55IiwiU0ZHTWFuYWdlciIsIlFDIiwiUHJvZHVjdGlvblN0b3JlSW5jaGFyZ2UiLCJWZWhpY2xlTWFuYWdlciIsIk1hbmFnZXIiLCJPcGVyYXRvciIsIldJUFN0b3JhZ2VMb2NhdGlvbiIsIk1hY2hpbmUgMiIsIk1hY2hpbmUgMyIsIkJBU0lDX0FDQ0VTUyJdLCJleHAiOjE3ODY1MTMyNjd9.6T2h7sVdWcA0p4iv1cWuSQJ6muGrnx6I59koOGTRFQ4"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/json"
}

response = requests.get(
    f"{BASE_URL}/api/v1/account/roles",
    headers=headers,
    verify=False
)

print("Status Code:", response.status_code)

if response.ok:
    print(response.json())
else:
    print(response.text)

Status Code: 200
[{'roleId': 1, 'roleName': 'SuperAdmin', 'displayName': 'Super Admin', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 2, 'roleName': 'GateKeeper', 'displayName': 'Gate Keeper', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 3, 'roleName': 'CustomManager', 'displayName': 'Custom Manager', 'createdBy': 'testuser', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 4, 'roleName': 'WareHouseManager', 'displayName': 'Ware House Manager', 'createdBy': 'test', 'createdDate': '2023-10-05T17:23:11.787', 'updatedDate': None, 'updatedBy': None, 'isActive': True}, {'roleId': 5, 'roleName': 'GateAdmin', 'displayName': 'Gate Admin', 'createdBy': 'test', 'createdDate': '2023-10-10T00:00:00', 'updatedDate': None, 'updatedBy': None, 'isActive': None

In [52]:
import requests
import pandas as pd
import json

swagger_url = "https://localhost:7204/swagger/v1/swagger.json"

swagger = requests.get(swagger_url, verify=False).json()

rows = []

for endpoint, methods in swagger["paths"].items():
    for method, details in methods.items():

        description = details.get("description")

        # If description is JSON, parse it
        try:
            description = json.loads(description) if description else None
        except Exception:
            pass

        rows.append({
            "Method": method.upper(),
            "Endpoint": endpoint,
            "Summary": details.get("summary"),
            "Description": description,          # JSON object or string
            "Authentication": "Authorize" in str(details.get("security", [])),
            "Request Body": "Yes" if "requestBody" in details else None,
            "Parameters": len(details.get("parameters", []))
        })

df = pd.DataFrame(rows)

In [ ]:
import requests
import pandas as pd
import json

swagger_url = "https://localhost:7204/swagger/v1/swagger.json"

swagger = requests.get(swagger_url, verify=False).json()

rows = []

for endpoint, methods in swagger["paths"].items():
    for method, details in methods.items():
        description = details.get("description")

        # If description is JSON, parse it
        try:
            description = json.loads(description) if description else None
        except Exception:
            pass

        if isinstance(description, (dict, list)):
            description_value = json.dumps(description)
        else:
            description_value = description

        rows.append({
            "Method": method.upper(),
            "Endpoint": endpoint,
            "Summary": details.get("summary"),
            "Description": description_value,
            "Authentication": "Authorize" in str(details.get("security", [])),
            "Request Body": "Yes" if "requestBody" in details else None,
            "Parameters": len(details.get("parameters", []))
        })


df = pd.DataFrame(rows)

print(df.head())

print(f"\nTotal APIs Found : {len(df)}")

# NOTE: This is a preview dataframe only. The schema of public.backend_api_table is different,
# so use the dedicated insertion cell below to write into the production table.


In [ ]:
import requests
import pandas as pd
import json
import uuid
import urllib3

from sqlalchemy import create_engine
from sqlalchemy.dialects.postgresql import JSONB

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ======================================================
# PostgreSQL Configuration
# ======================================================

DB_CONFIG = {
    "host": "localhost",
    "port": "5432",
    "database": "manufacturing_ai",
    "user": "postgres",
    "password": "0987654321"
}

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

# ======================================================
# Connection ID
# ======================================================

CONNECTION_ID = 1

# ======================================================
# Swagger
# ======================================================

BASE_URL = "https://localhost:7204"
SWAGGER_URL = f"{BASE_URL}/swagger/v1/swagger.json"

swagger = requests.get(
    SWAGGER_URL,
    verify=False
).json()

rows = []

# ======================================================
# Parse Swagger
# ======================================================

for endpoint, methods in swagger["paths"].items():

    for method, details in methods.items():

        # --------------------------
        # Read metadata from description
        # --------------------------

        metadata = {}

        try:
            metadata = json.loads(details.get("description", "{}"))
        except Exception:
            metadata = {}

        # --------------------------
        # API Group
        # --------------------------

        parts = endpoint.strip("/").split("/")

        api_group = parts[2] if len(parts) >= 3 else None

        api_version = (
            parts[1]
            if len(parts) > 1 and parts[1].startswith("v")
            else None
        )
        # --------------------------
        # Controller Name
        # --------------------------

        controller_name = api_group

        # --------------------------
        # Operation ID
        # --------------------------

        operation_id = details.get("operationId")

       
        # --------------------------
        # Parameters
        # --------------------------

        parameters = details.get("parameters", [])

        path_variables = []
        query_parameters = []
        required_fields = []
        optional_fields = []

        for p in parameters:

            if p.get("in") == "path":
                path_variables.append(p)

            elif p.get("in") == "query":
                query_parameters.append(p)

            if p.get("required"):
                required_fields.append(p.get("name"))
            else:
                optional_fields.append(p.get("name"))

        # --------------------------
        # Row
        # --------------------------

        rows.append({

            "api_id": str(uuid.uuid4()),

            "connection_id": CONNECTION_ID,

            "tool_name": metadata.get("tool_name"),

            "method": method.upper(),

            "end_point": endpoint,

            "description": metadata.get("description"),

            "parameters": parameters,

            "responses": responses,

            "authentication_required": bool(details.get("security")),

            "request_body": "requestBody" in details,

            "api_group": api_group,

            "controller_name": controller_name,

            "operation_id": operation_id,

            "required_fields": required_fields,

            "optional_fields": optional_fields,

            "agent_accessible": metadata.get(
                "agent_accessible",
                False
            ),

            "auto_register": metadata.get(
                "auto_register",
                True
            ),

            "canonical_entity": metadata.get(
                "canonical_entity"
            ),

            "api_version": api_version,

            "raw_metadata": metadata

        })

# ======================================================
# DataFrame
# ======================================================

df = pd.DataFrame(rows)

print(df.head())

print(f"\nTotal APIs Found : {len(df)}")

# ======================================================
# Remove old APIs for this connection
# ======================================================

from sqlalchemy import text

with engine.begin() as conn:
    conn.execute(
        text("""
            DELETE FROM backend_api_table
            WHERE connection_id = :id
        """),
        {"id": CONNECTION_ID}
    )
# ======================================================
# Upload
# ======================================================

df.to_sql(
    "backend_api_table",
    engine,
    if_exists="append",
    index=False,
    method="multi",
    dtype={

        "parameters": JSONB,

        "responses": JSONB,

        "path_variables": JSONB,

        "query_parameters": JSONB,

        "required_fields": JSONB,

        "optional_fields": JSONB,

        "request_schema": JSONB,

        "response_schema": JSONB,

        "http_status_codes": JSONB,

        "tags": JSONB,

        "raw_metadata": JSONB,

        "expected_output": JSONB

    }
)

print(f"\n✅ Successfully inserted {len(df)} APIs.")

                                 api_id  connection_id            tool_name  \
0  0e8c4e4c-7692-47c1-b9f9-00f8459e4abc              1           get_status   
1  0fb53ba1-3bca-4dde-bc8e-ff05aaa56671              1  create_user_profile   
2  d6ea0a7d-6a85-4216-87e2-4b837faa74b9              1            get_roles   
3  06a90ac7-fe61-4a62-9a5f-25c48cb8e049              1            get_menus   
4  826f6475-802f-4102-ae09-6e67fc2b8ba4              1           user_by_id   

  method                        end_point  \
0    GET                  /api/v1/account   
1   POST    /api/v1/account/user-creation   
2    GET            /api/v1/account/roles   
3    GET             /api/v1/account/menu   
4    GET  /api/v1/account/user-by-id/{id}   

                                         description  \
0   Executes GetStatus. Usable by AccountApi agents.   
1  Executes CreateUserProfile. Usable by AccountA...   
2    Executes GetRoles. Usable by AccountApi agents.   
3    Executes GetMenus. Usable

In [ ]:
import psycopg2
import json

try:
    conn = psycopg2.connect(
        dbname="manufacturing_ai",
        user="postgres",
        password="0987654321",
        host="localhost",
        port="5432"
    )
    cur = conn.cursor()

    # 1. Insert Template Pipelines into agentic_pipline
    insert_pipeline_query = """
        INSERT INTO public.agentic_pipline (
            agentic_id, system_prompt, tools_list, agent_desp, 
            required_app_access, active_status
        )
        VALUES (%s, %s, %s, %s, %s, %s);
    """
    
    pipelines_to_insert = [
        (
            'tpl_daily_ops_01', 
            'You are an Operations Assistant. Use your provided tools to fetch the latest work orders and machine statuses, then summarize the daily yield.', 
            ['{{GET_WORK_ORDERS_TOOL}}', '{{GET_MACHINE_STATUS_TOOL}}'], # Placeholders for tools
            'Template for Daily Operations Reporting', 
            'app_mes_01', 
            True
        ),
        (
            'tpl_iot_monitor_01', 
            'You are a Safety and Asset Monitoring Agent. Check the telemetry data and flag any anomalies in temperature or vibration.', 
            ['{{GET_TELEMETRY_TOOL}}'], 
            'Template for IoT Asset Monitoring', 
            'app_iot_01', 
            True
        )
    ]

    for pipeline in pipelines_to_insert:
        cur.execute(insert_pipeline_query, pipeline)

    # 2. Insert the corresponding execution steps into steps_in_pipline
    insert_steps_query = """
        INSERT INTO public.steps_in_pipline (
            step_id, agentic_id, prompt, human_in_loop_status, in_use_status
        )
        VALUES (%s, %s, %s, %s, %s);
    """
    
    steps_to_insert = [
        (
            'step_ops_fetch_01',
            'tpl_daily_ops_01',
            'Execute {{GET_WORK_ORDERS_TOOL}} to retrieve today''s production schedule. Then execute {{GET_MACHINE_STATUS_TOOL}} to check for downtime. Compile this into a summary.',
            False, # Autonomous data gathering
            True
        ),
        (
            'step_iot_check_01',
            'tpl_iot_monitor_01',
            'Run {{GET_TELEMETRY_TOOL}} on machine ID {{TARGET_MACHINE_ID}}. If temperature exceeds threshold, draft an alert.',
            True, # Requires human approval before sending alert
            True
        )
    ]

    for step in steps_to_insert:
        cur.execute(insert_steps_query, step)

    conn.commit()
    print("Successfully inserted template pipelines and steps.")

except Exception as e:
    print(f"An error occurred: {e}")
    if conn:
        conn.rollback()
finally:
    if conn:
        cur.close()
        conn.close()

Successfully inserted template pipelines and steps.
